# House Price Prediction with MLflow

In this notebook:
- Run a hyperparameter tuning process while training a model.
- Log every hyperparameter and metric in the MLflow UI.
- Compare the results of different runs in the MLflow UI.
- Select the best-performing run and register it in the MLflow Model Registry.

In [1]:
# Data manipulation and analysis
import pandas as pd
import numpy as np

# Data visualization
import matplotlib.pyplot as plt

# MLflow for experiment tracking and model logging
import mlflow
import mlflow.sklearn

# Machine Learning model
from sklearn.ensemble import RandomForestRegressor

# Utilities for splitting the dataset and hyperparameter tuning
from sklearn.model_selection import train_test_split, GridSearchCV

# Regression evaluation metric
from sklearn.metrics import mean_squared_error

# California Housing dataset
from sklearn.datasets import fetch_california_housing

c:\Users\swkra\OneDrive\Υπολογιστής\mlops\venv\Lib\site-packages\mlflow\__init__.py:41: UserWarning: Versions of mlflow (3.13.0) and child packages mlflow-tracing (3.14.0) are different. This may lead to unexpected behavior. Please install the same version of all MLflow packages.
  mlflow.mismatch._check_version_mismatch()


In [2]:
# Load the California Housing dataset
housing = fetch_california_housing()

# Preparing the dataset
data=pd.DataFrame(housing.data,columns=housing.feature_names)
data["Price"]=housing.target
data.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,Price
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


# Train test split, Model Hyperparameter Tuning, MLFLOW Experiments


In [3]:
from urllib.parse import urlparse

# Independent and Dependent Features
X=data.drop(columns=["Price"])
y=data["Price"]

In [ ]:
# Hyperparameter tuning
def hyperparameter_tunning(X_train,y_train,param_grid):
    rf=RandomForestRegressor()
    grid_search=GridSearchCV(estimator=rf,param_grid=param_grid, cv=3, n_jobs=1,verbose=2,scoring="neg_mean_squared_error")
    grid_search.fit(X_train,y_train)
    return grid_search

In [5]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

from mlflow.models import infer_signature

signature = infer_signature(X_train, y_train)

# Define the hyperparameters grid
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [5, 10, None],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}

In [6]:
# Connect to the MLflow Tracking Server
mlflow.set_tracking_uri("http://127.0.0.1:5000")

# Create or use an existing experiment
mlflow.set_experiment("Hyperparameter Search House Prediction")

# Start an MLflow run
with mlflow.start_run():

    # Perform hyperparameter tuning
    grid_search = hyperparameter_tunning(X_train, y_train, param_grid)

    # Get the best model
    best_model = grid_search.best_estimator_

    # Make predictions
    y_pred = best_model.predict(X_test)

    # Evaluate the model
    mse = mean_squared_error(y_test, y_pred)

    # Log the best hyperparameters
    mlflow.log_param("best_n_estimators", grid_search.best_params_["n_estimators"])
    mlflow.log_param("best_max_depth", grid_search.best_params_["max_depth"])
    mlflow.log_param("best_min_samples_split", grid_search.best_params_["min_samples_split"])
    mlflow.log_param("best_min_samples_leaf", grid_search.best_params_["min_samples_leaf"])

    # Log evaluation metric
    mlflow.log_metric("mse", mse)

    # Log and register the best model
    mlflow.sklearn.log_model(
        sk_model=best_model,
        artifact_path="model",
        signature=signature,
        registered_model_name="Best Random Forest Model"
    )

    print(f"Best Hyperparameters: {grid_search.best_params_}")
    print(f"Mean Squared Error: {mse}")

Fitting 3 folds for each of 24 candidates, totalling 72 fits
[CV] END max_depth=5, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=   4.5s
[CV] END max_depth=5, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=   4.4s
[CV] END max_depth=5, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=   4.2s
[CV] END max_depth=5, min_samples_leaf=1, min_samples_split=2, n_estimators=200; total time=   8.5s
[CV] END max_depth=5, min_samples_leaf=1, min_samples_split=2, n_estimators=200; total time=   8.5s
[CV] END max_depth=5, min_samples_leaf=1, min_samples_split=2, n_estimators=200; total time=   8.4s
[CV] END max_depth=5, min_samples_leaf=1, min_samples_split=5, n_estimators=100; total time=   4.1s
[CV] END max_depth=5, min_samples_leaf=1, min_samples_split=5, n_estimators=100; total time=   4.2s
[CV] END max_depth=5, min_samples_leaf=1, min_samples_split=5, n_estimators=100; total time=   4.2s
[CV] END max_depth=5, min_samples_leaf=

2026/08/02 20:34:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/08/02 20:34:22 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'Best Random Forest Model' already exists. Creating a new version of this model...
2026/08/02 20:35:09 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Best Random Forest Model, version 2
Created version '2' of model 'Best Random Forest Model'.


Best Hyperparameters: {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
Mean Squared Error: 0.25146443911283095
🏃 View run wistful-colt-682 at: http://127.0.0.1:5000/#/experiments/2/runs/0838bb4ae6ea4233b9e8ecebdb520e68
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


In [7]:
model_name = "Best Random Forest Model"
model_version = 2

model_uri = f"models:/{model_name}/{model_version}"

loaded_model = mlflow.sklearn.load_model(model_uri)